# Stage B validation — coverage + latency (§8.2.3)

Two views on gap-fill efficacy, evaluated for **both** Stage B candidates (RF and ST-kriging):
1. **Coverage audit** — per-month spatial coverage before and after gap-fill (§9 target: ≥ 95 %).
2. **SSO-stratified RMSE** — error vs *slots since last observed*.  Tests whether fill quality degrades gracefully as the latency widens (replaces the old DSO metric after the v3.4 30-min cadence pivot).

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


sys.path.insert(0, str(Path.cwd()))
import validate as vb
import config as cfg

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## Coverage audit — RF vs ST-kriging vs regression-kriging

Per-month fractions:
- `pre_fill_observed` — Stage A coverage before gap-fill (same for all candidates by construction).
- `post_fill_observed` — `is_observed` flag preserved through Stage B (should ≈ pre-fill).
- `post_fill_coverage` — total valid AOD coverage after gap-fill (observed + filled).

Run separately so the file readers walk each candidate tree.  Note `rf` fills
every cell (post-fill coverage = 1.0) by design; `kriging` leaves cells with no
ST neighbours as NaN; `rf_rk` should also reach 1.0 because its gap cells fall
back to the RF drift when the residual pool is empty.

In [ ]:
cov_rf = vb.coverage_audit(START, END, candidate='rf')
cov_kr = vb.coverage_audit(START, END, candidate='kriging')
cov_rk = vb.coverage_audit(START, END, candidate='rf_rk')
print('— RF —')
display(cov_rf)
print('— ST-kriging —')
display(cov_kr)
print('— RF + regression kriging —')
display(cov_rk)

### Side-by-side post-fill coverage

Merges the two audits on `month`.  `pre_fill_observed` is taken from one side (identical by construction); a discrepancy here would signal a Stage A → Stage B IO bug.

In [ ]:
cov_side = (cov_rf[['month', 'pre_fill_observed', 'post_fill_coverage']]
            .rename(columns={'post_fill_coverage': 'post_fill_rf'})
            .merge(cov_kr[['month', 'post_fill_coverage']]
                   .rename(columns={'post_fill_coverage': 'post_fill_kriging'}),
                   on='month')
            .merge(cov_rk[['month', 'post_fill_coverage']]
                   .rename(columns={'post_fill_coverage': 'post_fill_rf_rk'}),
                   on='month'))
cov_side['delta_rf_minus_kr']    = cov_side['post_fill_rf']    - cov_side['post_fill_kriging']
cov_side['delta_rf_rk_minus_kr'] = cov_side['post_fill_rf_rk'] - cov_side['post_fill_kriging']
cov_side

In [ ]:
if not cov_side.empty:
    ax = cov_side.plot(x='month',
                       y=['pre_fill_observed', 'post_fill_rf', 'post_fill_kriging',
                          'post_fill_rf_rk'],
                       kind='line', marker='o', figsize=(10, 4))
    ax.axhline(0.95, color='r', ls='--', lw=0.8, label='§9 target ≥95%')
    ax.set_ylim(0, 1.05); ax.set_ylabel('coverage fraction')
    ax.set_title('Stage B post-fill coverage — RF vs ST-kriging vs rf_rk')
    ax.legend(); ax.grid(alpha=0.3)

## SSO-stratified RMSE

For each AERONET-blind pair, count slot-steps back to the nearest observed slot at the same `(site, cell)`, including the nighttime gap if no earlier slot exists today.  RMSE binned by SSO answers: *does the fill quality degrade gracefully as the gap widens?*

In [ ]:
pairs_rf = vb.aeronet_pairs(START, END, candidate='rf', blind_only=True)
vb.sso_stratified_rmse(pairs_rf)

In [ ]:
pairs_rf = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=True)
vb.sso_stratified_rmse(pairs_rf)

In [ ]:
pairs_rk = vb.aeronet_pairs(START, END, candidate='rf_rk', blind_only=True)
vb.sso_stratified_rmse(pairs_rk)